# 03.2 — General parsers

The last notebook ended with a dictionary mapping file extensions to functions,
and a `parse()` that looked things up in it.

That has a name. You wrote a general document parser — in one notebook, badly.

Other people have written the same thing properly, and this notebook asks whether
you should throw yours away.

## The landscape

**MarkItDown** (Microsoft, MIT). A thin wrapper over format-specific libraries —
`pdfminer.six`, `mammoth`, `python-pptx`. No models, no GPU, installs in seconds.
Very wide format coverage, very fast, shallow.

**Unstructured**. Less a parser than a document ETL platform. Partitions files
into typed elements — `Title`, `NarrativeText`, `Table`, `ListItem` — each with
page numbers and coordinates. Chunking and connectors included.

**Docling** (LF AI & Data, MIT). Layout and table-structure models plus a vision
model for hard pages, all running locally. Best open-source table fidelity. Also
the heaviest install here.

Two more worth knowing about but not installing today: **Marker**, which is
GPU-first and produces excellent output under a GPL licence that needs a legal
check before commercial use, and **Apache Tika**, which is a JVM service with an
enormous format list and output that isn't RAG-shaped.

Install note: Docling is a large download. If you're on a slow connection, run
the first two and read the Docling results below.

In [1]:
!pip install -q markitdown[docx,pptx,xlsx,pdf]==0.1.7 \
               "unstructured[docx,pptx,xlsx,md]==0.27.5" \
               docling==2.126.0

## They all work

Point each one at the same file.

In [2]:
from pathlib import Path

CORPUS = Path('../../corpus/docs')
memo = CORPUS / 'sahel-hr-memo-2024-41-TRACKED.docx'

from markitdown import MarkItDown
from unstructured.partition.auto import partition
from docling.document_converter import DocumentConverter

markitdown = MarkItDown()
docling = DocumentConverter()


def by_markitdown(path):
    return markitdown.convert(str(path)).text_content


def by_unstructured(path):
    return '\n'.join(str(el) for el in partition(filename=str(path)))


def by_docling(path):
    return docling.convert(str(path)).document.export_to_markdown()


PARSERS = {
    'MarkItDown': by_markitdown,
    'Unstructured': by_unstructured,
    'Docling': by_docling,
}

for name, fn in PARSERS.items():
    text = fn(memo)
    print(f'{name:<14} {len(text):>6,} chars\n')
    print(f'Text:\n{text[:300]}\n')
    print("=" * 100)

MarkItDown      1,419 chars

Text:
# Memorandum: Revision of Staff Travel and Leave Entitlements

**To: All Staff**

From: Head, Human Capital

Date: 14 November 2024

Reference: SMB/HR/MEMO/2024/41

Following the review of staff welfare provisions approved by the Board Governance Committee on 29 October 2024, the following changes t

Unstructured    1,496 chars

Text:
Sahel Microfinance Bank Plc — Internal Memorandum
Memorandum: Revision of Staff Travel and Leave Entitlements
To: All Staff
From: Head, Human Capital
Date: 14 November 2024
Reference: SMB/HR/MEMO/2024/41
Following the review of staff welfare provisions approved by the Board Governance Committee on 2

Docling         1,424 chars

Text:
## Memorandum: Revision of Staff Travel and Leave Entitlements

**To: All Staff**

From: Head, Human Capital

Date: 14 November 2024

Reference: SMB/HR/MEMO/2024/41

Following the review of staff welfare provisions approved by the Board Governance Committee on 29 October 2024, the following

Three parsers, three plausible outputs, similar sizes. One line of code each
instead of the thirty you wrote.

If you stopped here you'd pick whichever installed fastest, and you would have
learned nothing. Character counts tell you a parser produced text. They don't
tell you whether it produced the *right* text.

## Test them against what you know is in there

This is the part that transfers to your own work. You can't evaluate a parser in
the abstract — you evaluate it against the specific things in your documents that
you know are hard.

Five checks, all from failures the last notebook exposed:

In [3]:
CHECKS = [
    ('DOCX   keeps the current figure',
     'sahel-hr-memo-2024-41-TRACKED.docx',   lambda t: '60,000' in t),
    
    ('DOCX   drops the deleted figure',
     'sahel-hr-memo-2024-41-TRACKED.docx',   lambda t: '35,000' not in t),

    ('XLSX   hidden sheet excluded',
     'kaduna-agro-distribution-2024.xlsx',   lambda t: '264100' not in t and '264.100' not in t),

    ('PPTX   speaker notes captured',
     'kaduna-agro-board-deck-2025-01.pptx',  lambda t: '2.4 billion' in t),

    ('HTML   navigation removed',
     'nfsc-circular-2025-02.html',           lambda t: 'Returns Portal' not in t),
]

header = f"{'check':<34}" + ''.join(f'{n:<15}' for n in PARSERS)
print(header)
print('-' * len(header))

for label, filename, test in CHECKS:
    row = f'{label:<34}'
    for name, fn in PARSERS.items():
        try:
            row += f"{'pass' if test(fn(CORPUS / filename))else 'FAIL':<15}"
        except Exception:
            row += f"{'error':<15}"
    print(row)

check                             MarkItDown     Unstructured   Docling        
-------------------------------------------------------------------------------
DOCX   keeps the current figure   pass           FAIL           pass           
DOCX   drops the deleted figure   pass           pass           pass           
XLSX   hidden sheet excluded      FAIL           FAIL           pass           


libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


PPTX   speaker notes captured     pass           FAIL           FAIL           
HTML   navigation removed         FAIL           pass           pass           


## Now the uncomfortable comparison

In [4]:
# The handlers from notebook 1, condensed. Nothing new — skip past it.
import csv, re, zipfile
from bs4 import BeautifulSoup
from lxml import etree
from openpyxl import load_workbook
from pptx import Presentation
from email import policy
from email.parser import BytesParser

W = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'


def read_docx(path):
    root = etree.fromstring(zipfile.ZipFile(path).read('word/document.xml'))
    for deleted in root.iter(f'{W}del'):
        deleted.getparent().remove(deleted)
    return '\n'.join(
        t for t in (''.join(x.text or '' for x in p.iter(f'{W}t'))
                    for p in root.iter(f'{W}p')) if t.strip())


def read_xlsx(path):
    wb = load_workbook(path, data_only=True)
    parts = []
    for sheet in wb.worksheets:
        if sheet.sheet_state != 'visible':
            continue
        parts.append(f'## {sheet.title}')
        for row in sheet.iter_rows():
            cells = [str(c.value) if c.value is not None else '' for c in row]
            if any(cells):
                parts.append(' | '.join(cells).rstrip(' |'))
    return '\n'.join(parts)


def read_pptx(path):
    parts = []
    for i, slide in enumerate(Presentation(path).slides, 1):
        body = [sh.text_frame.text for sh in slide.shapes
                if sh.has_text_frame and sh.text_frame.text.strip()]
        parts.append(f'## Slide {i}\n' + '\n'.join(body))
        if slide.has_notes_slide and slide.notes_slide.notes_text_frame.text.strip():
            parts.append('[speaker notes]\n'
                         + slide.notes_slide.notes_text_frame.text.strip())
    return '\n\n'.join(parts)


def read_html(path):
    soup = BeautifulSoup(path.read_text(encoding='utf-8'), 'lxml')
    for tag in soup(['nav', 'header', 'footer', 'aside', 'script', 'style', 'form']):
        tag.decompose()
    return (soup.find('main') or soup.body).get_text('\n', strip=True)


OURS = {'.docx': read_docx, '.xlsx': read_xlsx,
        '.pptx': read_pptx, '.html': read_html}


def parse(path):
    return OURS[Path(path).suffix.lower()](Path(path))


print(f"{'check':<34}ours")
print('-' * 40)
for label, filename, test in CHECKS:
    print(f"{label:<34}{'pass' if test(parse(CORPUS / filename)) else 'FAIL'}")

check                             ours
----------------------------------------
DOCX   keeps the current figure   pass
DOCX   drops the deleted figure   pass
XLSX   hidden sheet excluded      pass
PPTX   speaker notes captured     pass
HTML   navigation removed         pass


Five out of five. The one you wrote in an afternoon beats all three.

**That result is dishonest and you should know why.**

You wrote those handlers *after* being shown the failures. You knew the memo had
tracked changes, so you resolved them. You knew about the hidden sheet, so you
checked `sheet_state`. You knew the notes mattered, so you read them.

It isn't a better parser. It's a parser fitted to five known answers.

Point it at a corpus with traps you haven't seen — a legacy `.doc`, a password
-protected workbook, a PDF with a form layer, an email with an attachment that
matters — and it breaks immediately, while Docling handles several of them,
because Docling was built by people who have seen thousands of documents and you
have seen fifteen.

## So which do you use?

**Use a general parser.** They handle formats you haven't thought about, they're
maintained, and they encode a lot of accumulated knowledge about documents that
you don't have.

**Then test it against the things you know are hard in your corpus.** Not against
a benchmark — against your own documents, with checks you write because you know
what's in them.

That's what the three-parser comparison did. A `CHECKS` list and a loop, under
thirty lines, and it told you more about these tools than any feature list would
have. Write the equivalent for your own documents before you commit to a parser.

**Then override where it fails.** A general parser plus three custom handlers for
the cases that matter is a completely normal production setup. It is not an
admission of defeat.

Two of the five checks above aren't really parsing questions at all. Whether to
index a hidden sheet marked *DO NOT CIRCULATE*, and whether to index speaker
notes that say they're internal, are governance decisions. No parser can make
them for you, and a parser that silently picks one has picked wrong roughly half
the time.

## The decision for this course

We keep the hand-written handlers, for one reason: the rest of module 03 is about
cleaning, structure, metadata and provenance, and it's much easier to see those
steps when you can read the code that produces the text.

In a real project we'd start with Docling and override the speaker-notes case.

One caveat on that recommendation. We only tested Office and HTML files here.
Docling's real strength is PDF layout and table structure, which needs its vision
models — a much larger download, and a meaningfully different comparison from the
one above. If your corpus is mostly PDFs, run that comparison before choosing.

## What's next

Every parser above returned an empty string for the scanned document and none of
them mentioned it.

Notebook 3 is about detecting that before it reaches your index, along with the
rest of what arrives broken: OCR corruption, encoding damage, boilerplate, and
email threads that repeat themselves three times.